In [0]:
%sh
gunzip /Volumes/workspace/default/airlines-ml/202306/*.csv.gz
gunzip /Volumes/workspace/default/airlines-ml/202309/*.csv.gz
gunzip /Volumes/workspace/default/airlines-ml/202403/*.csv.gz

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

# 1. Root path pointing to your Databricks Volume structure
base_volume_path = "/Volumes/workspace/default/airlines-ml/*"

# 2. Dynamically load all flight data files using wildcards
flights_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"{base_volume_path}/Flights_*.csv")

# 3. Dynamically load all planned trajectory points files
points_filed_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"{base_volume_path}/Flight_Points_Filed_*.csv")

print(f"Consolidated Flights Row Count: {flights_df.count()}")

In [0]:
# 1. Convert string datetime fields into true Spark Timestamps
processed_flights = flights_df \
    .withColumn("filed_dep", F.to_timestamp("FILED OFF BLOCK TIME", "dd-MM-yyyy HH:mm:ss")) \
    .withColumn("actual_dep", F.to_timestamp("ACTUAL OFF BLOCK TIME", "dd-MM-yyyy HH:mm:ss"))

# 2. Calculate the difference in minutes
processed_flights = processed_flights.withColumn(
    "departure_delay_min", 
    (F.col("actual_dep").cast("long") - F.col("filed_dep").cast("long")) / 60
)

# 3. Establish the ground-truth binary target (1 = Delayed > 15 mins, 0 = On-Time)
processed_flights = processed_flights.withColumn(
    "is_delayed", 
    F.when(F.col("departure_delay_min") > 15, 1).otherwise(0)
)

# 4. Extract time features for seasonal analysis
processed_flights = processed_flights \
    .withColumn("month", F.month("filed_dep")) \
    .withColumn("hour", F.hour("filed_dep")) \
    .withColumn("day_of_week", F.dayofweek("filed_dep"))

In [0]:
# 1. Isolate the exact departure coordinate records (Sequence Number 0)
origin_nodes = points_filed_df.filter(F.col("Sequence Number") == 0) \
    .select(F.col("ECTRL ID").alias("points_id"), "Latitude", "Longitude")

# 2. Join back to the main flights table using a clean Inner Join
ml_ready_spark_df = processed_flights.join(
    origin_nodes, 
    processed_flights["ECTRL ID"] == origin_nodes["points_id"], 
    "inner"
).drop("points_id")

# 3. Retain only the features required for modeling, removing any incomplete rows
final_features_df = ml_ready_spark_df.select(
    "is_delayed", 
    "Latitude", 
    "Longitude", 
    "Requested FL", 
    "Actual Distance Flown (nm)", 
    "AC Operator",
    "month",
    "hour",
    "day_of_week"
).dropna()

In [0]:
# Extract a representative 20% sample to pass to the driver node memory
pandas_dataset = final_features_df.sample(withReplacement=False, fraction=0.2, seed=84).toPandas()

print(f"Pandas DataFrame successfully loaded. Matrix dimensions: {pandas_dataset.shape}")

In [0]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# 1. Isolate features from the target array
X = pandas_dataset.drop(columns=["is_delayed"])
y = pandas_dataset["is_delayed"]

# 2. Categorize input variables
num_cols = ["Latitude", "Longitude", "Requested FL", "Actual Distance Flown (nm)", "month", "hour", "day_of_week"]
cat_cols = ["AC Operator"]

# 3. Map out transformers using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric_scaling', StandardScaler(), num_cols),
        ('categorical_encoding', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ])

# 4. Perform a stratified split to maintain consistent seasonal delay proportions
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=84, 
    stratify=y
)

# 5. Fit and transform the data partitions
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print(f"Data ready for training. Train shape: {X_train_transformed.shape} | Test shape: {X_test_transformed.shape}")

In [0]:
# Cell 6: Training the Baseline Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier

# Initialize the RandomForest copy matching the project's strategy
# n_estimators can be increased later (e.g., 100 or 200) for better stability
rf_model = RandomForestClassifier(
    n_estimators=100, 
    max_depth=15, 
    random_state=84, 
    n_jobs=-1
)

# Fit the model onto the transformed training tensors
print("Training the Random Forest model...")
rf_model.fit(X_train_transformed, y_train)
print("Model trained successfully!")

In [0]:
# Cell 7: Model Evaluation and Performance Metrics
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score

# Execute predictions on the test dataset split
y_pred = rf_model.predict(X_test_transformed)

# Generate and display the operational Confusion Matrix
conf_matrix = confusion_matrix(y_test, y_pred)
print("--- Confusion Matrix ---")
print(conf_matrix)

# Print complete classification metrics focusing heavily on Precision
print("\n--- Performance Report ---")
print(classification_report(y_test, y_pred, target_names=['On-Time (0)', 'Delayed (1)']))

In [0]:
# Cell 8: Training an Alternative Gradient Boosting Model (XGBoost)
# Note: Ensure xgboost is installed in your Databricks Cluster (%pip install xgboost)
%pip install xgboost
from xgboost import XGBClassifier

# Initialize the XGBoost model optimized for binary classification
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=1, # Adjust if you need to handle severe class imbalances
    random_state=84,
    n_jobs=-1
)

# Train the gradient boosted trees
print("Training the XGBoost model...")
xgb_model.fit(X_train_transformed, y_train)

# Evaluate XGBoost predictions
y_pred_xgb = xgb_model.predict(X_test_transformed)
print("\n--- XGBoost Performance Report ---")
print(classification_report(y_test, y_pred_xgb, target_names=['On-Time (0)', 'Delayed (1)']))